In [1]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import warnings
 
warnings.filterwarnings("ignore")
 
# ---------- configuration ----------
DEFAULT_N_TOP = 10
DATE_COL = "Date"
TIMESTAMP_COL = "Timestamp"
 
 
# ---------- data loading ----------
def load_tvl_data(path: str) -> pd.DataFrame:
    """
    Load daily TVL (Total Value Locked) data per protocol/chain.
 
    - Parses a date column to a DatetimeIndex.
    - Drops the raw timestamp column if present.
    - Fills missing values with zero.
 
    Assumes: one date column and the rest are numeric TVL columns.
    """
    df = pd.read_csv(path)
 
    if DATE_COL not in df.columns:
        raise ValueError(f"Expected date column '{DATE_COL}' in CSV.")
 
    df[DATE_COL] = pd.to_datetime(df[DATE_COL]).dt.strftime('%Y-%m-%d')
    df.set_index(DATE_COL, inplace=True)
    df.index.name = "date"
 
    if TIMESTAMP_COL in df.columns:
        df = df.drop(columns=TIMESTAMP_COL)
 
    # Ensure numeric dtype for TVL columns
    df = df.apply(pd.to_numeric, errors="coerce")
 
    # Missing TVL treated as zero
    df = df.fillna(0)
 
    return df
 
 
# ---------- selection ----------
def get_top_n_protocols(tvl_df: pd.DataFrame, n: int = DEFAULT_N_TOP) -> pd.Index:
    """
    Rank protocols by their average TVL over the full period
    and return the top n, sorted from largest to smallest.
    """
    average_values = tvl_df.mean(axis=0)  # column-wise mean
    top_n_protocols = average_values.nlargest(n).sort_values(ascending=False).index
    return top_n_protocols
 
 
# ---------- plotting ----------
def plot_top_tvl_stacked_area_plotly(
    tvl_df: pd.DataFrame,
    top_protocols: pd.Index,
    save_path: str | None = None,
    figsize: tuple[int, int] = (2000, 1200),
) -> go.Figure:
    """
    Interactive stacked area chart using Plotly.
    Largest protocol plotted on top of the stack.
    """
    
    # Initialize the figure
    fig = make_subplots(rows=1, cols=1, shared_xaxes=True, vertical_spacing=0)
    
    fig.update_layout(
        height=figsize[1], 
        width=figsize[0], 
        title_text='', 
        showlegend=True, 
        font=dict(family='Times New Roman', size=24),
        plot_bgcolor='rgba(0,0,0,0)',  
        paper_bgcolor='rgba(0,0,0,0)',  
        font_color='black'
    )
    
    # Define a custom color palette
    custom_colors = (px.colors.qualitative.Dark24 + 
                     px.colors.qualitative.Light24 + 
                     px.colors.qualitative.Alphabet)
    
    # Plot the top N protocols in reverse order (largest at top)
    # top_protocols is already sorted largest-to-smallest
    # We reverse it so we plot from smallest to largest (stack from bottom to top)
    for i, protocol in enumerate(top_protocols[::-1]):
        color = custom_colors[i % len(custom_colors)]
        
        fig.add_trace(go.Scatter(
            y=tvl_df[protocol], 
            x=tvl_df.index,  
            fill='tonexty',
            name=protocol,
            line=dict(color=color, width=4)
        ), row=1, col=1)
    
    fig.update_xaxes(
        showline=True, 
        linewidth=1, 
        linecolor='black', 
        mirror=True, 
        showgrid=False, 
        title='',
        dtick='M3',
        tickformat='%b\n%Y',
        showticklabels=True
    )
    
    fig.update_yaxes(
        title_text='TVL (in USD)',  
        showline=True, 
        linewidth=1, 
        linecolor='black', 
        mirror=True, 
        showgrid=False
    )
    
    if save_path is not None:
        fig.write_image(save_path, width=figsize[0], height=figsize[1], scale=2)
    
    fig.show()
    return fig

In [2]:
tvl_df = load_tvl_data("../Data/chains.csv")
top_protocols = get_top_n_protocols(tvl_df, n=DEFAULT_N_TOP)
plot_top_tvl_stacked_area_plotly(tvl_df, top_protocols, save_path="tvl_top10.png")